Question 1: Image Preprocessing for Inference (PyTorch)

Write a function to load an image and preprocess it for inference. 

In [4]:
from PIL import Image
from torchvision import transforms

def preprocess_image_for_inference(image_path, target_size=(224, 224)):
       
    # 1. Load and basic prep
    image = Image.open(image_path).convert("RGB")
    
    # 2. Define standard preprocessing pipeline
    preprocess = transforms.Compose([
        transforms.Resize(target_size),           # Resize maintaining aspect ratio
        transforms.CenterCrop(target_size),       # Crop to exact size
        transforms.ToTensor(),                    # PIL → tensor (H,W,C) → (C,H,W) [0,1]
        transforms.Normalize(                     # ImageNet stats (critical!)
            mean=[0.485, 0.456, 0.406],          # R,G,B channels
            std=[0.229, 0.224, 0.225]
        )
    ])
    
    # 3. Apply transforms + batch dimension
    input_tensor = preprocess(image).unsqueeze(0)  # (1, C, H, W)
    
    return input_tensor


 Question 2: Predict on New Image with a Trained Model

 Perform prediction and get the class label.

In [11]:
import torch
from torchvision import models

model = models.resnet50()
model.eval()

image_path = r'D:\HopeAI\Assignments\6 DeepLearningWeek11\1.Deep_Learning_Question_Set1\4.Coding\Images_to_use\flower.jpg'

input_image = preprocess_image_for_inference(image_path)
with torch.no_grad():
    output = model(input_image)
    predicted_class = output.argmax(1).item()
print("Predicted Class:", predicted_class)


Predicted Class: 865


 Question 3: Build a CNN to classify CIFAR-10 images (PyTorch)

 Create a CNN model that classifies images from the CIFAR-10 dataset with accuracy above 60%.

In [23]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

# Preprocessing
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)

# CNN model
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.fc = nn.Sequential(
            nn.Linear(64*8*8, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

net = Net()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.001)

# Training 

for epoch in range(5):
    net.train()
    for images, labels in trainloader:
        outputs = net(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    # Eval each epoch
    net.eval()
    train_correct = sum((net(images).argmax(1) == labels).sum().item() for images, labels in trainloader)
    test_correct = sum((net(images).argmax(1) == labels).sum().item() for images, labels in testloader)
    
    print(f"Epoch {epoch+1}: Train {100.*train_correct/50000:.1f}%, Test {100.*test_correct/10000:.1f}%")




Epoch 1: Train 63.8%, Test 61.8%
Epoch 2: Train 70.7%, Test 66.8%
Epoch 3: Train 77.1%, Test 70.4%
Epoch 4: Train 81.1%, Test 72.0%
Epoch 5: Train 82.2%, Test 71.8%


 Question 4: Identify Overfitting from Training Logs and Solve It

 Problem: You notice the training accuracy increases but validation accuracy stagnates. Modify the model using dropout and early stopping(use mnist dataset)

In [24]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0
y_train, y_test = to_categorical(y_train), to_categorical(y_test)

model = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(10, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
early_stop = EarlyStopping(patience=3, restore_best_weights=True)

model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs=10, callbacks=[early_stop])

train_loss, train_acc = model.evaluate(x_train, y_train, verbose=0)
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)

print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy:  {test_acc:.4f}")


d:\HopeAI\.objdet_ls\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step


d:\HopeAI\.objdet_ls\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.9108 - loss: 0.2941 - val_accuracy: 0.9637 - val_loss: 0.1186
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.9561 - loss: 0.1451 - val_accuracy: 0.9713 - val_loss: 0.0917
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9657 - loss: 0.1144 - val_accuracy: 0.9743 - val_loss: 0.0874
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9700 - loss: 0.0997 - val_accuracy: 0.9734 - val_loss: 0.0869
Epoch 5/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9732 - loss: 0.0874 - val_accuracy: 0.9773 - val_loss: 0.0734
Epoch 6/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9757 - loss: 0.0796 - val_accuracy: 0.9804 - val_loss: 0.0655
Epoch 7/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9776 - loss: 0.0720 - val_accuracy: 0.9776 - val_loss: 0.0771
Epoch 8/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9796 - loss: 0.0661 - 

 Question 5: Transfer Learning with Pretrained VGG16 (Cats vs Dogs)

 Problem: Use VGG16 for binary classification with fine-tuning

In [26]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Dropout

base_model = VGG16(include_top=False, input_shape=(224, 224, 3), weights='imagenet')
for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = Flatten()(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
